In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')
real_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(real_path)
print(f"Shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_drop=df.drop('Order_ID', axis=1)
df_drop.info()

In [ ]:
# Task 2: Write your code here: Wrote in several cells


In [ ]:
# Task 2.1: Analyze missing values
missing_percentage = (df_drop.isnull().sum() / len(df_drop)) * 100
missing_data = pd.DataFrame({
'Column': missing_percentage.index,
'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)
print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Task 2.2: Delete Missing data as the persentage of missing data is samll we can delete the rows wtih missig data
df_clean = df_drop.copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level','Time_of_Day','Courier_Experience_yrs'])
print(f"After dropping missing: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df_clean)

In [ ]:
df_clean.shape

In [ ]:
# Task 4: Write your code here:
# Encode type columns
df_encoded = df_clean.copy()
le = LabelEncoder()
categ_col=['Weather','Vehicle_Type','Time_of_Day','Traffic_Level']
for col in categ_col:
    df_encoded[col] = le.fit_transform(df_clean[col])

df_encoded.head()

In [ ]:
# Task 5: Write your code here:
target_column = "Delivery_Time"
X = df_encoded.drop(target_column, axis=1)
y = df_encoded[target_column]
scaler = StandardScaler()
X_scale = scaler.fit_transform(X)


print(X_scale.shape)
print(y.shape)

In [ ]:
# Task 6: Write your code here:
# Don't needed
# print(y)

In [ ]:
# Task 1: Write your code here:
# the data splitted into X (features) , Y (Target) before scaling to avoid scaling the target Y
X = X_scale
y = y
print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)



In [ ]:
# Task 2,3,4,5: Write your code here:
# Define Model
from sklearn.model_selection import  KFold

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
for train_idx, test_idx in kf.split(X):
  X_train, X_test = X[train_idx], X[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
  # Train model
  model.fit(X_train, y_train)
  # Predict
  y_pred = model.predict(X_test)
  # Evaluation metrics
  mae_scores.append(mean_absolute_error(y_test, y_pred))

# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)

print(f"MAE : {np.mean(mae_scores):.2f}")
print("-"*40)

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols=['Distance_km'	,'Weather',	'Traffic_Level'	,'Time_of_Day'	,'Vehicle_Type'	,'Preparation_Time_min'	,'Courier_Experience_yrs']
importance = pd.DataFrame({
  'feature': feature_cols,
  'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('predicted delivery time Distribution')
plt.xlabel('predicted delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: